In [1]:
pip install pandas selenium webdriver-manager google-search-results

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [3]:
import time
import pandas as pd
from serpapi import GoogleSearch
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime

SERP_API_KEY = "2f16e7d005997e32f72d022af06744466adc45ea331f88ae0be84a1dca0e9311"


def build_hbl_dawn_query():

    keywords = [
        '"Habib Bank Limited"',
        '"HBL Pakistan"',
        '"HBL Bank"',
        '"HBL stock"',
        '"HBL share"',
        '"HBL shares"',
        '"HBL earnings"',
        '"HBL.KA"'
    ]
    return " OR ".join(keywords)


def scrape_dawn_news_with_dates(max_results_per_year=100, output_file="Dawn_HBL.csv"):
    site_name = "DAWN"
    site_domain = "dawn.com"
    date_class = "timestamp--date"

    print(f"\n🔎 Scraping {site_name} ({site_domain}) for the past 10 years...")

    hbl_query = build_hbl_dawn_query()
    all_articles = []
    current_year = datetime.now().year

    for year in range(current_year - 10, current_year + 1):
        start_date = f"01/01/{year}"
        end_date = f"12/31/{year}"
        print(f"\n📅 Year: {year}")

        for start in range(0, max_results_per_year, 10):
            params = {
                "engine": "google",
                "q": f"{hbl_query} site:{site_domain}",
                "location": "Pakistan",
                "hl": "en",
                "gl": "pk",
                "api_key": SERP_API_KEY,
                "num": 10,
                "start": start,
                "tbs": f"cdr:1,cd_min:{start_date},cd_max:{end_date}"
            }

            try:
                search = GoogleSearch(params)
                results = search.get_dict()
                organic = results.get("organic_results", [])

                if not organic:
                    break

                for result in organic:
                    all_articles.append({
                        "source": site_name,
                        "title": result.get("title"),
                        "url": result.get("link"),
                        "snippet": result.get("snippet", ""),
                        "published_date": None,
                        "query": "HBL_STRICT",
                        "year": year
                    })

                time.sleep(1)

            except Exception as e:
                print(f"⚠️ Error fetching results for {year}: {e}")
                break

    df = pd.DataFrame(all_articles)
    df.drop_duplicates(subset="url", inplace=True)
    df.reset_index(drop=True, inplace=True)

    if df.empty:
        print("⚠️ No results found.")
        return df

    print(f"\n📰 Found {len(df)} unique DAWN articles. Extracting publish dates...")

    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )

    # 🔒 STRICT DATE EXTRACTION — timestamp--date ONLY
    for index, url in enumerate(df["url"]):
        try:
            driver.get(url)
            time.sleep(1.5)

            try:
                date_element = driver.find_element(By.CLASS_NAME, date_class)
                date_text = date_element.text.replace("Published", "").strip()
                df.at[index, "published_date"] = date_text
                print(f"[{index}] ✅ {date_text}")
            except:
                df.at[index, "published_date"] = "Not Found"
                print(f"[{index}] ⚠️ Date not found")

        except Exception as e:
            print(f"[{index}] ❌ Error fetching {url}: {e}")
            df.at[index, "published_date"] = "Error"

    driver.quit()

    df.to_csv(output_file, index=False)
    print(f"\n✅ Done! Collected {len(df)} HBL articles from DAWN.")
    print(f"📁 Saved to: {output_file}")

    return df


if __name__ == "__main__":
    scrape_dawn_news_with_dates()



🔎 Scraping DAWN (dawn.com) for the past 10 years...

📅 Year: 2015

📅 Year: 2016

📅 Year: 2017

📅 Year: 2018

📅 Year: 2019

📅 Year: 2020

📅 Year: 2021

📅 Year: 2022

📅 Year: 2023

📅 Year: 2024

📅 Year: 2025

📰 Found 410 unique DAWN articles. Extracting publish dates...
[0] ⚠️ Date not found
[1] ⚠️ Date not found
[2] ⚠️ Date not found
[3] ⚠️ Date not found
[4] ⚠️ Date not found
[5] ⚠️ Date not found
[6] ⚠️ Date not found
[7] ⚠️ Date not found
[8] ⚠️ Date not found
[9] ⚠️ Date not found
[10] ⚠️ Date not found
[11] ⚠️ Date not found
[12] ⚠️ Date not found
[13] ⚠️ Date not found
[14] ⚠️ Date not found
[15] ⚠️ Date not found
[16] ⚠️ Date not found
[17] ⚠️ Date not found
[18] ⚠️ Date not found
[19] ⚠️ Date not found
[20] ⚠️ Date not found
[21] ⚠️ Date not found
[22] ⚠️ Date not found
[23] ⚠️ Date not found
[24] ⚠️ Date not found
[25] ⚠️ Date not found
[26] ⚠️ Date not found
[27] ⚠️ Date not found
[28] ⚠️ Date not found
[29] ⚠️ Date not found
[30] ⚠️ Date not found
[31] ⚠️ Date not found
[32]

In [2]:
import time
import pandas as pd
from serpapi import GoogleSearch
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime

# ==========================
# CONFIG
# ==========================
SERP_API_KEY = "2f16e7d005997e32f72d022af06744466adc45ea331f88ae0be84a1dca0e9311"

TEST_MODE = True
TEST_ARTICLE_LIMIT = 10     # 🔒 Max articles to fetch in test mode
MAX_RESULTS_PER_YEAR = 10   # 🔒 One page only to save credits

# ==========================
# STRICT HBL QUERY
# ==========================
def build_hbl_dawn_query():
    return (
        '"Habib Bank Limited" OR '
        '"HBL Pakistan" OR '
        '"HBL Bank" OR '
        '"HBL stock" OR '
        '"HBL share" OR '
        '"HBL shares" OR '
        '"HBL earnings" OR '
        '"HBL.KA"'
    )


# ==========================
# MAIN SCRAPER
# ==========================
def scrape_dawn_news_with_dates(output_file="dawn_hbl_test.csv"):
    site_name = "DAWN"
    site_domain = "dawn.com"
    date_class = "timestamp--date"

    print("\n🔎 TEST MODE ENABLED — Fetching first 10 articles only")

    hbl_query = build_hbl_dawn_query()
    all_articles = []
    article_count = 0

    current_year = datetime.now().year

    for year in range(current_year - 10, current_year + 1):
        if TEST_MODE and article_count >= TEST_ARTICLE_LIMIT:
            break

        start_date = f"01/01/{year}"
        end_date = f"12/31/{year}"

        print(f"\n📅 Year: {year}")

        for start in range(0, MAX_RESULTS_PER_YEAR, 10):
            if TEST_MODE and article_count >= TEST_ARTICLE_LIMIT:
                break

            params = {
                "engine": "google",
                "q": f"{hbl_query} site:{site_domain}",
                "location": "Pakistan",
                "hl": "en",
                "gl": "pk",
                "api_key": SERP_API_KEY,
                "num": 10,
                "start": start,
                "tbs": f"cdr:1,cd_min:{start_date},cd_max:{end_date}"
            }

            try:
                search = GoogleSearch(params)
                results = search.get_dict()
                organic = results.get("organic_results", [])

                if not organic:
                    break

                for result in organic:
                    if TEST_MODE and article_count >= TEST_ARTICLE_LIMIT:
                        break

                    all_articles.append({
                        "source": site_name,
                        "title": result.get("title"),
                        "url": result.get("link"),
                        "snippet": result.get("snippet", ""),
                        "published_date": None,
                        "query": "HBL_STRICT",
                        "year": year
                    })

                    article_count += 1

                time.sleep(1)

            except Exception as e:
                print(f"⚠️ Error fetching results for {year}: {e}")
                break

    df = pd.DataFrame(all_articles).drop_duplicates(subset="url").reset_index(drop=True)

    if df.empty:
        print("⚠️ No results found.")
        return df

    print(f"\n📰 Collected {len(df)} articles. Extracting publish dates...")

    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )

    for index, url in enumerate(df["url"]):
        try:
            driver.get(url)
            time.sleep(1.2)

            try:
                date_element = driver.find_element(By.CLASS_NAME, "timestamp--date")
                date_text = date_element.text.replace("Published", "").strip()
                df.at[index, "published_date"] = date_text
                print(f"[{index}] ✅ {date_text}")
            except:
                df.at[index, "published_date"] = "Not Found"
                print(f"[{index}] ⚠️ Date not found")

            if date_element:
                df.at[index, "published_date"] = (
                    date_element.text.replace("Published", "").strip()
                )
                print(f"[{index}] ✅ {df.at[index, 'published_date']}")
            else:
                df.at[index, "published_date"] = "Not Found"
                print(f"[{index}] ⚠️ Date not found")

        except Exception as e:
            print(f"[{index}] ❌ Error fetching {url}: {e}")
            df.at[index, "published_date"] = "Error"

    driver.quit()

    df.to_csv(output_file, index=False)
    print(f"\n✅ TEST COMPLETE — Saved to: {output_file}")

    return df


# ==========================
# RUN
# ==========================
if __name__ == "__main__":
    scrape_dawn_news_with_dates()



🔎 TEST MODE ENABLED — Fetching first 10 articles only

📅 Year: 2015

📰 Collected 10 articles. Extracting publish dates...
[0] ⚠️ Date not found
[1] ⚠️ Date not found
[2] ⚠️ Date not found
[3] ⚠️ Date not found
[4] ⚠️ Date not found
[5] ⚠️ Date not found
[6] ⚠️ Date not found
[7] ⚠️ Date not found
[8] ⚠️ Date not found
[9] ⚠️ Date not found

✅ TEST COMPLETE — Saved to: dawn_hbl_test.csv
